In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install gdown

In [2]:
import zipfile
import gdown

In [5]:
import gdown

url = "https://drive.google.com/uc?id=1JClBTqsqlADo7SA7xIQTebAos0-KJQdt"
gdown.download(url, "Base_dataset.zip", quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1JClBTqsqlADo7SA7xIQTebAos0-KJQdt
From (redirected): https://drive.google.com/uc?id=1JClBTqsqlADo7SA7xIQTebAos0-KJQdt&confirm=t&uuid=48bf94df-c932-4b81-aedb-b49cf753f60c
To: /kaggle/working/Base_dataset.zip
100%|██████████| 704M/704M [00:07<00:00, 92.6MB/s] 


'Base_dataset.zip'

In [6]:
with zipfile.ZipFile("Base_dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

In [ ]:
import os, json, torch, numpy as np
from PIL import Image, ImageDraw
from torch.utils.data import Dataset

CLASS_MAP = {
        "laptop":0,
         "mouse":1,
         "keyboard":2,
          "mobile":3,
          "monitor":4}

class LabelmeDataset(Dataset):

    def __init__(self, root, transforms=None):
        self.root = root
        self.transforms = transforms

        self.images = sorted([
            f for f in os.listdir(root)
            if f.endswith((".png",".jpg",".jpeg"))
        ])

    def __len__(self):
        return len(self.images)

    def polygon_to_mask(self, points, h, w):
        mask = Image.new("L",(w,h),0)
        ImageDraw.Draw(mask).polygon(
            [tuple(p) for p in points],
            outline=1,
            fill=1
        )
        return np.array(mask,dtype=np.uint8)

    def __getitem__(self, idx):

        img_name = self.images[idx]
        img_path = os.path.join(self.root,img_name)

        json_path = os.path.splitext(img_path)[0] + ".json"

        img = Image.open(img_path).convert("RGB")
        w,h = img.size

        with open(json_path) as f:
            ann = json.load(f)

        boxes=[]
        labels=[]
        masks=[]

        for obj in ann["shapes"]:

            points=np.array(obj["points"])
            label=obj["label"]

            xmin=np.min(points[:,0])
            xmax=np.max(points[:,0])
            ymin=np.min(points[:,1])
            ymax=np.max(points[:,1])

            boxes.append([xmin,ymin,xmax,ymax])
            labels.append(CLASS_MAP[label])

            mask=self.polygon_to_mask(points,h,w)
            masks.append(mask)

        boxes=torch.as_tensor(boxes,dtype=torch.float32)
        labels=torch.as_tensor(labels,dtype=torch.int64)
        masks=torch.as_tensor(
            np.array(masks),
            dtype=torch.uint8
        )

        area=(boxes[:,3]-boxes[:,1])*(boxes[:,2]-boxes[:,0])

        target={
            "boxes":boxes,
            "labels":labels,
            "masks":masks,
            "image_id":torch.tensor([idx]),
            "area":area,
            "iscrowd":torch.zeros(
                (len(labels),),dtype=torch.int64
            )
        }

        img=np.array(img)
        img=torch.tensor(
            img.transpose(2,0,1),
            dtype=torch.float32
        )/255.

    from torch.utils.data import DataLoader

def collate_fn(batch):
    return tuple(zip(*batch))

dataset=LabelmeDataset("Base_dataset")

loader=DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
) img,target




In [ ]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    return tuple(zip(*batch))

dataset=LabelmeDataset("Base_dataset")

loader=DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

In [ ]:
import torchvision
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

num_classes = 4  # background + 3 classes

model = torchvision.models.detection.maskrcnn_resnet50_fpn(
    weights="DEFAULT"
)

# box classifier
in_features=model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor=FastRCNNPredictor(
    in_features,
    num_classes
)

# mask head
in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
hidden_layer=256

model.roi_heads.mask_predictor=MaskRCNNPredictor(
    in_features_mask,
    hidden_layer,
    num_classes
)

In [ ]:
import os
import torch

device = (torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))

model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# -----------------------------
# Resume + Checkpoint settings
# -----------------------------
checkpoint_path = "maskrcnn_checkpoint.pth"
best_model_path = "best_maskrcnn.pth"

start_epoch = 0
best_loss = float("inf")

# Early stopping patience
patience = 5
patience_counter = 0

epochs = 50


# -----------------------------
# Resume if checkpoint exists
# -----------------------------
if os.path.exists(checkpoint_path):
    print("Resuming from checkpoint...")

    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])

    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_loss = checkpoint["best_loss"]
    patience_counter = checkpoint.get("patience_counter",0)

    print(f"Resumed at epoch {start_epoch}")


# -----------------------------
# Training
# -----------------------------
for epoch in range(start_epoch, epochs):

    model.train()
    total_loss = 0

    for images, targets in loader:

        images = [img.to(device) for img in images]

        targets = [{k:v.to(device) for k,v in t.items()} for t in targets ]

        loss_dict = model(images, targets)

        loss = sum( v for v in loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()


    epoch_loss = total_loss / len(loader)

    print(f"Epoch {epoch+1} " f"Loss: {epoch_loss:.5f}" )


    # -------------------------
    # Save resume checkpoint
    # -------------------------
    torch.save({
            "epoch":epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_loss": best_loss,
            "patience_counter": patience_counter},
        checkpoint_path
    )


    # -------------------------
    # Early stopping logic
    # -------------------------
    if epoch_loss < best_loss:

        print( f"Improved " f"{best_loss:.5f}" f" -> {epoch_loss:.5f}")

        best_loss = epoch_loss
        patience_counter = 0

        # save best model
        torch.save(
            model.state_dict(),
            best_model_path
        )

    else:
        patience_counter += 1
        print(f"No improvement " f"({patience_counter}/{patience})")
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

In [ ]:
model.eval()

pred=model([dataset[0][0].to(device)])
print(pred)